In [1]:
import pandas as pd
import requests
import time

df = pd.read_csv("1_power_plants_kerala.csv")

FEATURES = [
    "ALLSKY_SFC_SW_DWN",
    "CLRSKY_SFC_SW_DWN",
    "ALLSKY_SFC_SW_DNI",
    "ALLSKY_SFC_SW_DIFF",
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "PRECTOTCORR",
    "WS2M",
    "WS10M",
    "PS"
]

def fetch_nasa(lat, lon):
    url = (
        "https://power.larc.nasa.gov/api/temporal/climatology/point"
        f"?parameters={','.join(FEATURES)}"
        f"&community=RE"
        f"&longitude={lon}&latitude={lat}"
        "&format=JSON"
    )
    
    r = requests.get(url)
    data = r.json()["properties"]["parameter"]
    
    result = {}
    
    for f in FEATURES:
        # Annual mean value
        result[f] = data[f]["ANN"]
    
    return result


rows = []

for _, row in df.iterrows():
    try:
        nasa = fetch_nasa(row["latitude"], row["longitude"])
        
        combined = {
            "plant_name": row["plant_name"],
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "capacity_mw": row["installed_capacity_mw"],
            "production_gwh": row["annual_estimated_production_gwh"]
        }
        
        combined.update(nasa)
        rows.append(combined)
        
        time.sleep(1)  # avoid API rate limit
        
    except Exception as e:
        print("Error for:", row["plant_name"], e)

final_df = pd.DataFrame(rows)

print(final_df.head())

                                 plant_name  latitude  longitude  capacity_mw  \
0                      Kasaragod Solar Park   12.5000    75.0000         50.0   
1  Cochin International Airport Solar Plant   10.1520    76.4019         40.0   
2           Kayamkulam Floating Solar Plant    9.1700    76.5000          0.5   
3  Rajiv Gandhi Indoor Stadium Solar System    9.9312    76.2673          1.0   
4              Kannur Airport Solar Project   11.9200    75.5500          2.5   

   production_gwh  ALLSKY_SFC_SW_DWN  CLRSKY_SFC_SW_DWN  ALLSKY_SFC_SW_DNI  \
0            75.0             5.0378             6.6010             3.1433   
1            60.0             5.0827             6.5222             3.0084   
2             0.7             5.2452             6.5710             3.3194   
3             1.4             5.2452             6.5710             3.3194   
4             3.5             5.2282             6.5179             3.4963   

   ALLSKY_SFC_SW_DIFF    T2M  T2M_MAX  T2M_M

In [4]:
final_df.to_csv("power_plants_withfeatures.csv")

In [5]:
import pandas as pd
import requests
import time

# Load your dataset
df = pd.read_csv("1_power_plants_kerala.csv")

# NASA features (complete solar + weather set)
FEATURES = [
    "ALLSKY_SFC_SW_DWN",   # GHI (solar radiation)
    "CLRSKY_SFC_SW_DWN",   # clear sky radiation
    "ALLSKY_SFC_SW_DNI",   # direct irradiance
    "ALLSKY_SFC_SW_DIFF",  # diffuse radiation
    "T2M",                 # temperature
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",                # humidity
    "PRECTOTCORR",        # rainfall
    "WS2M",               # wind speed
    "WS10M",
    "PS"                  # pressure
]

def fetch_nasa_data(lat, lon):
    url = (
        "https://power.larc.nasa.gov/api/temporal/climatology/point"
        f"?parameters={','.join(FEATURES)}"
        f"&community=RE"
        f"&longitude={lon}&latitude={lat}"
        "&format=JSON"
    )

    r = requests.get(url, timeout=30)
    data = r.json()["properties"]["parameter"]

    result = {}
    for f in FEATURES:
        result[f] = data[f]["ANN"]  # annual mean

    return result


results = []

for _, row in df.iterrows():
    print(f"Fetching: {row['plant_name']}")

    try:
        nasa = fetch_nasa_data(row["latitude"], row["longitude"])

        merged = {
            "plant_name": row["plant_name"],
            "location": row["location"],
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "capacity_mw": row["installed_capacity_mw"],
            "production_gwh": row["annual_estimated_production_gwh"]
        }

        merged.update(nasa)
        results.append(merged)

        time.sleep(1)  # avoid API rate limit

    except Exception as e:
        print("Error:", row["plant_name"], e)

final_df = pd.DataFrame(results)

final_df.to_csv("2022_2024.csv")

Fetching: Kasaragod Solar Park
Fetching: Cochin International Airport Solar Plant
Fetching: Kayamkulam Floating Solar Plant
Fetching: Rajiv Gandhi Indoor Stadium Solar System
Fetching: Kannur Airport Solar Project
Fetching: Technopark Solar Installation
Fetching: KSEB Kasaragod Rooftop Cluster
Fetching: Idukki Solar Hybrid Pilot
Fetching: Alappuzha Canal Solar Pilot
Fetching: Trivandrum International Airport Solar


In [ ]:
import pandas as pd
import requests
import time

df = pd.read_csv("1_power_plants_kerala.csv")

FEATURES = [
    "T2M",
    "RH2M",
    "PRECTOTCORR",
    "WS2M",
    "ALLSKY_SFC_SW_DWN"
]

def fetch_year_data(lat, lon, start, end):
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters={','.join(FEATURES)}"
        f"&community=RE"
        f"&longitude={lon}&latitude={lat}"
        f"&start={start}&end={end}"
        "&format=JSON"
    )

    r = requests.get(url, timeout=60)
    data = r.json()["properties"]["parameter"]

    df_list = []

    for feature in FEATURES:
        series = data[feature]
        df_temp = pd.DataFrame(list(series.items()), columns=["date", feature])
        
        if len(df_list) == 0:
            df_list = df_temp
        else:
            df_list = df_list.merge(df_temp, on="date")

    return df_list


all_data = []

for _, row in df.iterrows():
    print("Processing:", row["plant_name"])

    try:
        data_2022 = fetch_year_data(row["latitude"], row["longitude"], "20220101", "20221231")
        data_2024 = fetch_year_data(row["latitude"], row["longitude"], "20240101", "20241231")

        data_2022["year"] = 2022
        data_2024["year"] = 2024

        data_2022["plant_name"] = row["plant_name"]
        data_2024["plant_name"] = row["plant_name"]

        all_data.append(data_2022)
        all_data.append(data_2024)

        time.sleep(1)

    except Exception as e:
        print("Error:", row["plant_name"], e)

final_df = pd.concat(all_data, ignore_index=True)

print(final_df.head())

Processing: Kasaragod Solar Park
Processing: Cochin International Airport Solar Plant
Processing: Kayamkulam Floating Solar Plant
Processing: Rajiv Gandhi Indoor Stadium Solar System
Processing: Kannur Airport Solar Project
Processing: Technopark Solar Installation
Processing: KSEB Kasaragod Rooftop Cluster
Processing: Idukki Solar Hybrid Pilot
Processing: Alappuzha Canal Solar Pilot
Processing: Trivandrum International Airport Solar
       date    T2M   RH2M  PRECTOTCORR  WS2M  ALLSKY_SFC_SW_DWN  year  \
0  20220101  27.18  75.07         0.37  1.94             4.8086  2022   
1  20220102  26.96  73.87         0.17  1.53             5.7336  2022   
2  20220103  26.74  70.92         0.05  1.56             5.6462  2022   
3  20220104  26.45  68.36         0.02  1.45             5.6357  2022   
4  20220105  26.08  72.43         0.06  1.09             5.3717  2022   

             plant_name  
0  Kasaragod Solar Park  
1  Kasaragod Solar Park  
2  Kasaragod Solar Park  
3  Kasaragod Solar 

In [7]:
final_df.to_csv("2022_2024.csv")

In [9]:
df=pd.read_csv("all_plant_generation_clean_v2.csv")
df.columns

Index(['Date', 'Station', 'State', 'Sector', 'Owner', 'Type', 'Capacity_MW',
       'Daily_Generation_MU', 'Monthly_Generation_MU'],
      dtype='object')

In [10]:
df["Station"].unique()

array(['RENEW SURYA RAVI PVT LTD', 'RENEW SURYA ROSHNI PVT LTD',
       'RENEW SURYA VIHAAN PRIVATE LIMITED',
       'RISING SUN ENERGY (K) PVT LTD', 'SB ENERGY FOUR PVT LTD',
       'SB ENERGY SIX PRIVATE LTD.', 'TATA POWER GREEN ENERGY LIMITED',
       'TATA POWER RENEWABLE ENERGY LTD', 'TATA POWER SAURYA LIMITED',
       'THAR SURYA 1 PRIVATE LIMITED', 'TRANSITION RENEWABLE ENERGY',
       'ADANI HYBRID ENERGY JAISALMER FOUR LTD SOLAR',
       'ADANI HYBRID ENERGY JAISALMER FOUR LTD WIND',
       'ADANI HYBRID ENERGY JAISALMER ONE LIMITED SOLAR',
       'ADANI HYBRID ENERGY JAISALMER ONE LIMITED WIND',
       'ADANI HYBRID ENERGY JAISALMER THREE LIMITED',
       'ADANI HYBRID ENERGY JAISALMER THREE LIMITED WIND',
       'ADANI HYBRID ENERGY JAISALMER TWO LIMITED',
       'ADANI HYBRID ENERGY JAISALMER TWO LIMITED WIND',
       'ADANI JAISALMER ONE SEPL SOLAR', 'ADANI JAISALMER ONE SEPL WIND',
       'SERENTICA RENEWABLES INDIA 4 PRIVATE LIMITED_BKN2',
       'TRANSITION GREEN ENERGY